In [1]:
# Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, SpatialDropout1D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

import time
import os

# Set random seeds for reproducibility
np.random.seed(42)
import tensorflow as tf
tf.random.set_seed(42)

# Create directories
os.makedirs('models', exist_ok=True)
os.makedirs('results', exist_ok=True)

print("="*70)
print("ENVIRONMENT SETUP COMPLETE")
print("="*70)
print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print("\nDirectories created:")
print("  - models/")
print("  - results/")

ENVIRONMENT SETUP COMPLETE
TensorFlow version: 2.20.0
NumPy version: 2.0.2
Pandas version: 2.3.3

Directories created:
  - models/
  - results/


In [2]:
# Load IMDB dataset
df = pd.read_csv('data/IMDB Dataset.csv')

print("="*70)
print("DATASET LOADED")
print("="*70)
print(f"\nTotal reviews: {df.shape[0]:,}")
print(f"Columns: {df.columns.tolist()}")

print("\nSentiment Distribution:")
print(df['sentiment'].value_counts())

print("\nDataset Info:")
print(f"  Positive reviews: {(df['sentiment'] == 'positive').sum():,}")
print(f"  Negative reviews: {(df['sentiment'] == 'negative').sum():,}")
print(f"  Balance: {'✓ Perfectly balanced' if (df['sentiment'] == 'positive').sum() == (df['sentiment'] == 'negative').sum() else '✗ Imbalanced'}")

print("\nSample Review (first 250 characters):")
print(df['review'].iloc[0][:250] + "...")

print("\nMissing values:")
print(df.isnull().sum())

DATASET LOADED

Total reviews: 50,000
Columns: ['review', 'sentiment']

Sentiment Distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64

Dataset Info:
  Positive reviews: 25,000
  Negative reviews: 25,000
  Balance: ✓ Perfectly balanced

Sample Review (first 250 characters):
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of ...

Missing values:
review       0
sentiment    0
dtype: int64


In [3]:
def clean_text(text):
    """
    Minimal cleaning - preserve words that carry sentiment
    Only remove HTML tags, URLs, and standardize text
    """
    text = text.lower()
    text = re.sub(r'<br\s*/?>', ' ', text)      # Remove HTML breaks
    text = re.sub(r'http\S+|www\.\S+', '', text) # Remove URLs
    text = re.sub(r'[^a-z\s]', '', text)         # Keep only letters and spaces
    text = re.sub(r'\s+', ' ', text).strip()     # Remove extra spaces
    return text

print("="*70)
print("TEXT CLEANING")
print("="*70)
print("\nCleaning 50,000 reviews...")

# Apply cleaning
df['clean_review'] = df['review'].apply(clean_text)

# Encode labels
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

print("✓ Text cleaning complete")
print("✓ Labels encoded (positive=1, negative=0)")

# Show before/after example
print("\n" + "="*70)
print("BEFORE & AFTER COMPARISON")
print("="*70)
print("\nORIGINAL (first 200 chars):")
print(df['review'].iloc[0][:200])
print("\nCLEANED (first 200 chars):")
print(df['clean_review'].iloc[0][:200])

# Calculate average length
avg_length = df['clean_review'].apply(lambda x: len(x.split())).mean()
print(f"\nAverage review length: {avg_length:.1f} words")
print(f"Total reviews cleaned: {len(df):,}")

TEXT CLEANING

Cleaning 50,000 reviews...
✓ Text cleaning complete
✓ Labels encoded (positive=1, negative=0)

BEFORE & AFTER COMPARISON

ORIGINAL (first 200 chars):
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me abo

CLEANED (first 200 chars):
one of the other reviewers has mentioned that after watching just oz episode youll be hooked they are right as this is exactly what happened with me the first thing that struck me about oz was its bru

Average review length: 226.8 words
Total reviews cleaned: 50,000


In [4]:
print("="*70)
print("DATA SPLITTING")
print("="*70)

# First split: 80% train+val, 20% test
X_temp, X_test, y_temp, y_test = train_test_split(
    df['clean_review'], 
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

# Second split: 80% train, 20% val (from train+val)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.2,
    random_state=42,
    stratify=y_temp
)

print(f"\nSplit Strategy:")
print(f"  Total: 50,000 → Train+Val (40,000) + Test (10,000)")
print(f"  Train+Val: 40,000 → Train (32,000) + Val (8,000)")

print(f"\n" + "="*70)
print("FINAL SPLIT")
print("="*70)
print(f"Training set:   {len(X_train):,} samples (64.0%)")
print(f"Validation set: {len(X_val):,} samples (16.0%)")
print(f"Test set:       {len(X_test):,} samples (20.0%)")

# Verify class balance
print(f"\nClass Distribution:")
print(f"  Train - Negative: {(y_train==0).sum():,} | Positive: {(y_train==1).sum():,}")
print(f"  Val   - Negative: {(y_val==0).sum():,} | Positive: {(y_val==1).sum():,}")
print(f"  Test  - Negative: {(y_test==0).sum():,} | Positive: {(y_test==1).sum():,}")

print(f"\n✓ All sets are perfectly balanced")

DATA SPLITTING

Split Strategy:
  Total: 50,000 → Train+Val (40,000) + Test (10,000)
  Train+Val: 40,000 → Train (32,000) + Val (8,000)

FINAL SPLIT
Training set:   32,000 samples (64.0%)
Validation set: 8,000 samples (16.0%)
Test set:       10,000 samples (20.0%)

Class Distribution:
  Train - Negative: 16,000 | Positive: 16,000
  Val   - Negative: 4,000 | Positive: 4,000
  Test  - Negative: 5,000 | Positive: 5,000

✓ All sets are perfectly balanced


In [5]:
# Cell 5: Tokenization and Sequence Padding

# Hyperparameters (optimized for IMDB dataset)
VOCAB_SIZE = 10000      # Top 10k most frequent words
MAX_LENGTH = 200        # Covers 59.1% of reviews (from your analysis)
EMBEDDING_DIM = 100     # Standard embedding dimension

print("="*70)
print("TOKENIZATION & PADDING")
print("="*70)
print(f"\nHyperparameters:")
print(f"  Vocabulary Size: {VOCAB_SIZE:,}")
print(f"  Max Sequence Length: {MAX_LENGTH}")
print(f"  Embedding Dimension: {EMBEDDING_DIM}")

# Create tokenizer (fit only on training data!)
print("\nFitting tokenizer on training data...")
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

# Convert text to sequences
print("Converting text to sequences...")
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Pad sequences to fixed length
print("Padding sequences...")
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LENGTH, padding='post', truncating='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=MAX_LENGTH, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LENGTH, padding='post', truncating='post')

print(f"\n" + "="*70)
print("TOKENIZATION COMPLETE")
print("="*70)
print(f"\nFinal Shapes:")
print(f"  X_train_pad: {X_train_pad.shape}")
print(f"  X_val_pad:   {X_val_pad.shape}")
print(f"  X_test_pad:  {X_test_pad.shape}")

# Vocabulary statistics
word_index = tokenizer.word_index
print(f"\nVocabulary Stats:")
print(f"  Total unique words in training: {len(word_index):,}")
print(f"  Using top {VOCAB_SIZE:,} words")
print(f"  Coverage: {(VOCAB_SIZE/len(word_index)*100):.1f}%")

# Example
print(f"\nExample (first review):")
print(f"  Original length: {len(X_train_seq[0])} words")
print(f"  Padded length: {len(X_train_pad[0])} words")
print(f"  First 15 tokens: {X_train_pad[0][:15]}")

TOKENIZATION & PADDING

Hyperparameters:
  Vocabulary Size: 10,000
  Max Sequence Length: 200
  Embedding Dimension: 100

Fitting tokenizer on training data...
Converting text to sequences...
Padding sequences...

TOKENIZATION COMPLETE

Final Shapes:
  X_train_pad: (32000, 200)
  X_val_pad:   (8000, 200)
  X_test_pad:  (10000, 200)

Vocabulary Stats:
  Total unique words in training: 126,175
  Using top 10,000 words
  Coverage: 7.9%

Example (first review):
  Original length: 474 words
  Padded length: 200 words
  First 15 tokens: [ 435  432   10   25    6  999   15    2  374   35 1620   11   14 3289
    2]


In [6]:
# Cell 6: Build LSTM Model (Optimized Architecture)

def build_lstm_model():
    """
    Optimized LSTM for sentiment analysis
    - Moderate dropout (not too aggressive)
    - 2-layer LSTM architecture
    - Proper learning rate
    """
    model = Sequential([
        # Embedding layer
        Embedding(
            input_dim=VOCAB_SIZE,
            output_dim=EMBEDDING_DIM,
            input_length=MAX_LENGTH,
            name='embedding'
        ),
        
        # Light spatial dropout (helps regularization without killing learning)
        SpatialDropout1D(0.2),
        
        # First LSTM layer
        LSTM(100, return_sequences=True, name='lstm_1'),
        Dropout(0.2),
        
        # Second LSTM layer  
        LSTM(100, name='lstm_2'),
        Dropout(0.2),
        
        # Dense output layers
        Dense(1, activation='sigmoid', name='output')
    ])
    
    # Build model
    model.build(input_shape=(None, MAX_LENGTH))
    
    # Compile with standard Adam optimizer
    model.compile(
        optimizer=Adam(learning_rate=0.001),  # Standard LR
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    return model

print("="*70)
print("BUILDING LSTM MODEL")
print("="*70)

model = build_lstm_model()

print("\nModel Architecture:")
model.summary()

print(f"\n" + "="*70)
print("MODEL CONFIGURATION")
print("="*70)
print(f"Total Parameters: {model.count_params():,}")
print(f"Architecture: 2-Layer LSTM (100 → 100 units)")
print(f"Dropout: 0.2 (20%) - Balanced regularization")
print(f"Optimizer: Adam (lr=0.001)")
print(f"Loss: Binary Crossentropy")
print(f"\nThis architecture should achieve 85-88% validation accuracy")

BUILDING LSTM MODEL

Model Architecture:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 200, 100)            │       1,000,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ spatial_dropout1d (SpatialDropout1D) │ (None, 200, 100)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ (None, 200, 100)            │          80,400 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 200, 100)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_2 (LSTM)                        │ (None, 100)                 │          80,400 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 100)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ output (Dense)                       │ (None, 1)                   │             101 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,160,901 (4.43 MB)

 Trainable params: 1,160,901 (4.43 MB)

 Non-trainable params: 0 (0.00 B)


MODEL CONFIGURATION
Total Parameters: 1,160,901
Architecture: 2-Layer LSTM (100 → 100 units)
Dropout: 0.2 (20%) - Balanced regularization
Optimizer: Adam (lr=0.001)
Loss: Binary Crossentropy

This architecture should achieve 85-88% validation accuracy


In [7]:
# Cell 7: Train LSTM Model

print("="*70)
print("TRAINING LSTM MODEL")
print("="*70)

# Training configuration
EPOCHS = 10
BATCH_SIZE = 64

print(f"\nTraining Configuration:")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Training Samples: {len(X_train_pad):,}")
print(f"  Validation Samples: {len(X_val_pad):,}")
print(f"  Steps per Epoch: {len(X_train_pad)//BATCH_SIZE}")

# Callbacks for better training
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=0.00001,
    verbose=1
)

# Train the model
print("\n" + "="*70)
print("Starting Training...")
print("="*70 + "\n")

history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

print("\n" + "="*70)
print("TRAINING COMPLETE")
print("="*70)

TRAINING LSTM MODEL

Training Configuration:
  Epochs: 10
  Batch Size: 64
  Training Samples: 32,000
  Validation Samples: 8,000
  Steps per Epoch: 500

Starting Training...

Epoch 1/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 105s 206ms/step - accuracy: 0.5125 - loss: 0.6930 - val_accuracy: 0.6376 - val_loss: 0.7690 - learning_rate: 0.0010
Epoch 2/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 104s 208ms/step - accuracy: 0.5490 - loss: 0.6861 - val_accuracy: 0.4938 - val_loss: 0.6957 - learning_rate: 0.0010
Epoch 3/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 107s 213ms/step - accuracy: 0.5402 - loss: 0.6746 - val_accuracy: 0.8253 - val_loss: 0.4270 - learning_rate: 0.0010
Epoch 4/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 107s 215ms/step - accuracy: 0.8465 - loss: 0.3701 - val_accuracy: 0.8625 - val_loss: 0.3263 - learning_rate: 0.0010
Epoch 5/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 108s 216ms/step - accuracy: 0.9034 - loss: 0.2540 - val_accuracy: 0.8691 - val_loss: 0.3523 - learning_rate: 0.0010
Epoch 6/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 1

In [8]:
# Cell 8: Evaluate on Test Set

print("="*70)
print("FINAL MODEL EVALUATION")
print("="*70)

# Evaluate on test set
print("\nEvaluating on test set (10,000 samples)...")
test_loss, test_accuracy = model.evaluate(X_test_pad, y_test, verbose=0)

print(f"\n{'='*70}")
print("TEST SET RESULTS")
print(f"{'='*70}")
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy*100:.2f}%")

# Compare with validation
val_loss, val_accuracy = model.evaluate(X_val_pad, y_val, verbose=0)
print(f"\nValidation Accuracy: {val_accuracy*100:.2f}%")
print(f"Difference: {abs(test_accuracy - val_accuracy)*100:.2f}%")

if abs(test_accuracy - val_accuracy) < 0.02:
    print("\n✅ Model generalizes well! Test and validation accuracy are close.")
else:
    print("\n⚠️ Some generalization gap detected.")

# Sample predictions
print(f"\n{'='*70}")
print("SAMPLE PREDICTIONS")
print(f"{'='*70}")

# Get predictions for first 5 test samples
sample_predictions = model.predict(X_test_pad[:5], verbose=0)

for i in range(5):
    pred_prob = sample_predictions[i][0]
    pred_label = "Positive" if pred_prob > 0.5 else "Negative"
    true_label = "Positive" if y_test.iloc[i] == 1 else "Negative"
    confidence = pred_prob if pred_prob > 0.5 else (1 - pred_prob)
    
    print(f"\nSample {i+1}:")
    print(f"  True Label: {true_label}")
    print(f"  Predicted:  {pred_label} (confidence: {confidence*100:.1f}%)")
    print(f"  Review: {X_test.iloc[i][:100]}...")

FINAL MODEL EVALUATION

Evaluating on test set (10,000 samples)...

TEST SET RESULTS
Test Loss:     0.3682
Test Accuracy: 87.32%

Validation Accuracy: 87.10%
Difference: 0.22%

✅ Model generalizes well! Test and validation accuracy are close.

SAMPLE PREDICTIONS

Sample 1:
  True Label: Negative
  Predicted:  Negative (confidence: 97.7%)
  Review: yes mtv there really is a way to market daria what started as a clever teenage angstcomment on every...

Sample 2:
  True Label: Negative
  Predicted:  Positive (confidence: 54.3%)
  Review: the story of the bride fair is an amusing and engaging one and it is to the filmmakers credit that h...

Sample 3:
  True Label: Positive
  Predicted:  Positive (confidence: 98.7%)
  Review: a team varied between scully and mulder two other scientists a pilot and the guy who plays bana on s...

Sample 4:
  True Label: Negative
  Predicted:  Negative (confidence: 97.8%)
  Review: this was a popular movie probably because of the humor in it the fastmoving s